# Lake Urmia case study

In [66]:
# imports
import ee
import re
import pandas as pd
import leafmap.maplibregl as leafmap
import urllib.request
import os
import imageio
import glob
from IPython.display import display, HTML

from PIL import Image, ImageDraw, ImageFont
import imageio
import numpy as np
import os

In [2]:
ee.Authenticate()
ee.Initialize(project="alphaearth-476700")

In [12]:
roi = ee.Geometry.Rectangle([45.0, 37.0, 46.0, 38.5])

In [13]:
embeddings = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL') \
    .filterDate('2017-01-01', '2018-01-01') \
    .filterBounds(roi) \
    .mosaic().clip(roi)

In [24]:
# show embeddings

# Visualization: pick 3 embedding bands for RGB
vis_params = {
    'bands': ['A02', 'A16', 'A09'],  # example bands for visualization
    'min': -0.3,
    'max': 0.3
}

# Create a map
# coordinates need to be passed as lon, lat
m = leafmap.Map(center=[45.5, 37.5], zoom=8)
m.add_ee_layer(embeddings, vis_params, 'Embeddings')

In [25]:
m

Html(children=[<leafmap.maplibregl.Map object at 0x000001DD44D4A190>, Card(children=[Btn(children=[Icon(childr…

In [ ]:
# Lake Urmia AOI
roi = ee.Geometry.Rectangle([45.0, 37.0, 46.0, 38.5])

years = list(range(2017, 2025))

vis_params = {
    "bands": ["A02", "A16", "A09"],
    "min": -0.3,
    "max": 0.3
}

# Folder where PNGs will be saved
out_dir = "../data/urmia_frames"
os.makedirs(out_dir, exist_ok=True)

# --------------------------
# Export frames (PNG)
# --------------------------
frames = []

for year in years:
    print(f"Processing {year}...")
    
    img = (
        ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
        .filterDate(f"{year}-01-01", f"{year+1}-01-01")
        .filterBounds(roi)
        .mosaic()
        .clip(roi)
        .visualize(**vis_params)
    )
    
    # Get a URL for the image
    url = img.getThumbURL({
        "region": roi,
        "dimensions": 512,   # or 720, 1080, etc.
        "format": "png"
    })
    
    # Download the PNG
    png_path = os.path.join(out_dir, f"urmia_{year}.png")
    
    urllib.request.urlretrieve(url, png_path)

    frames.append(png_path)

print("All frames downloaded!")

Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
All frames downloaded!


# Generate embeddings .gif

In [ ]:
gif_path = "urmia_embeddings_smooth.gif"

# Parameters
frame_repeat = 10       # number of times each frame is repeated
fade_steps = 5          # number of interpolation frames between consecutive years
frame_duration = 1.0    # seconds per repeated frame (total duration = frame_repeat * frame_duration)

output_frames = []

# Prepare base frames with year titles
base_frames = []
for png_path in frames:
    year = png_path.split("_")[-1].split(".")[0]
    img = Image.open(png_path).convert("RGBA")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("arial.ttf", 40)
    except:
        font = ImageFont.load_default()
    draw.text((20, 20), f"Lake Urmia: {year}",
              fill=(255, 255, 255, 255),
              stroke_width=2,
              stroke_fill=(0, 0, 0, 255))
    base_frames.append(img.convert("RGB"))

# Add repeated frames and fade transitions
for i in range(len(base_frames)):
    # Repeat current frame
    for _ in range(frame_repeat):
        output_frames.append(base_frames[i])
    
    # Add fade to next frame (skip for last frame)
    if i < len(base_frames) - 1:
        next_frame = np.array(base_frames[i+1], dtype=np.float32)
        curr_frame = np.array(base_frames[i], dtype=np.float32)
        for t in range(1, fade_steps+1):
            alpha = t / (fade_steps + 1)
            interp_frame = (1-alpha)*curr_frame + alpha*next_frame
            interp_img = Image.fromarray(np.uint8(interp_frame))
            output_frames.append(interp_img)

# Save GIF
imageio.mimsave(gif_path, output_frames, duration=frame_duration)

print(f"Smooth GIF saved as {gif_path}")

Smooth GIF saved as urmia_embeddings_smooth.gif


# Unsupervised learning with k-means

Train on 2017 and ifnerence on 2017-2024 for results stability

In [ ]:
img1 = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL') \
    .filterDate('2017-01-01', '2018-01-01') \
    .filterBounds(roi) \
    .mosaic().clip(roi)

samples = img1.sample(
    region=roi,
    scale=20,
    numPixels=5000,
    geometries=True  # keep geometry to map clusters later
)

# Create the clusterer
k=5
clusterer = ee.Clusterer.wekaKMeans(nClusters=k)

# Train with number of clusters (k) passed in train() method
clusterer = clusterer.train(
    features=samples,
    inputProperties=img1.bandNames()
)

# Apply the clusterer to the whole image
result = img1.cluster(clusterer)

In [ ]:
# -----------------------------
# Years to process
# -----------------------------
years = list(range(2017, 2025))

# -----------------------------
# Cluster parameters
# -----------------------------
k = 5
palette = ['#ff0000','#00ff00','#0000ff','#ffff00','#800080']  # length = k

# Output folder for PNGs
out_dir = "../data/urmia_clusters"
os.makedirs(out_dir, exist_ok=True)

# -----------------------------
# Loop over years
# -----------------------------
frames = []

for year in years:
    print(f"Processing {year}...")
    
    # Load the embedding image for the year
    img = ee.ImageCollection('GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL') \
        .filterDate(f'{year}-01-01', f'{year+1}-01-01') \
        .filterBounds(roi) \
        .mosaic().clip(roi)
    
    # Apply clustering
    result = img.cluster(clusterer).rename('cluster')
    
    # Visualize clusters
    vis_params = {
        'min': 0,
        'max': k-1,
        'palette': palette
    }
    
    # Create PNG thumbnail
    url = result.visualize(**vis_params).getThumbURL({
        "region": roi,
        "dimensions": 512,   # or 720, 1080, etc.
        "format": "png"
    })
    
    png_path = os.path.join(out_dir, f"urmia_cluster_{year}.png")
    urllib.request.urlretrieve(url, png_path)
    frames.append(png_path)
    
print("All yearly cluster PNGs downloaded!")

Processing 2017...
Processing 2018...
Processing 2019...
Processing 2020...
Processing 2021...
Processing 2022...
Processing 2023...
Processing 2024...
All yearly cluster PNGs downloaded!


In [61]:
frames = glob.glob('../data/urmia_clusters/*.png')

In [63]:
gif_path = "urmia_embeddings_cluster_smooth.gif"

# Parameters
frame_repeat = 10       # number of times each frame is repeated
fade_steps = 5          # number of interpolation frames between consecutive years
frame_duration = 1.0    # seconds per repeated frame (total duration = frame_repeat * frame_duration)

output_frames = []

# Prepare base frames with year titles
base_frames = []
for png_path in frames:
    year = png_path.split("_")[-1].split(".")[0]
    img = Image.open(png_path).convert("RGBA")
    draw = ImageDraw.Draw(img)
    try:
        font = ImageFont.truetype("arial.ttf", 40)
    except:
        font = ImageFont.load_default()
    draw.text((20, 20), f"Lake Urmia: {year}",
              fill=(255, 255, 255, 255),
              stroke_width=2,
              stroke_fill=(0, 0, 0, 255))
    base_frames.append(img.convert("RGB"))

# Add repeated frames and fade transitions
for i in range(len(base_frames)):
    # Repeat current frame
    for _ in range(frame_repeat):
        output_frames.append(base_frames[i])
    
    # Add fade to next frame (skip for last frame)
    if i < len(base_frames) - 1:
        next_frame = np.array(base_frames[i+1], dtype=np.float32)
        curr_frame = np.array(base_frames[i], dtype=np.float32)
        for t in range(1, fade_steps+1):
            alpha = t / (fade_steps + 1)
            interp_frame = (1-alpha)*curr_frame + alpha*next_frame
            interp_img = Image.fromarray(np.uint8(interp_frame))
            output_frames.append(interp_img)

# Save GIF
imageio.mimsave(gif_path, output_frames, duration=frame_duration)

print(f"Smooth GIF saved as {gif_path}")

Smooth GIF saved as urmia_embeddings_cluster_smooth.gif


# Joint view

In [67]:
from PIL import Image, ImageDraw, ImageFont
import imageio
import numpy as np

gif_path = "urmia_embeddings_cluster_smooth_joint.gif"

# Parameters
frame_repeat = 10       # number of times each frame is repeated
fade_steps = 5          # number of interpolation frames between consecutive years
frame_duration = 1.0    # seconds per repeated frame

output_frames = []

# Assume you have two lists of PNG paths, same order: embeddings and clusters
# Example:
# embeddings_frames = ["data/urmia_embeddings_2017.png", "data/urmia_embeddings_2018.png", ...]
# cluster_frames = ["data/urmia_cluster_2017.png", "data/urmia_cluster_2018.png", ...]

base_frames = []

embeddings_frames = glob.glob('../data/urmia_frames/*.png')
cluster_frames = glob.glob('../data/urmia_clusters/*.png')

for emb_path, clus_path in zip(embeddings_frames, cluster_frames):
    year = emb_path.split("_")[-1].split(".")[0]
    
    # Open both images
    emb_img = Image.open(emb_path).convert("RGBA")
    clus_img = Image.open(clus_path).convert("RGBA")
    
    # Make a new image side by side
    width, height = emb_img.width + clus_img.width, max(emb_img.height, clus_img.height)
    combined = Image.new("RGBA", (width, height))
    combined.paste(emb_img, (0,0))
    combined.paste(clus_img, (emb_img.width,0))
    
    # Draw title on top-left
    draw = ImageDraw.Draw(combined)
    try:
        font = ImageFont.truetype("arial.ttf", 40)
    except:
        font = ImageFont.load_default()
    draw.text((20, 20), f"Lake Urmia: {year}", fill=(255,255,255,255),
              stroke_width=2, stroke_fill=(0,0,0,255))
    
    base_frames.append(combined.convert("RGB"))

# Add repeated frames and fade transitions
for i in range(len(base_frames)):
    # Repeat current frame
    for _ in range(frame_repeat):
        output_frames.append(base_frames[i])
    
    # Add fade to next frame (skip for last frame)
    if i < len(base_frames) - 1:
        next_frame = np.array(base_frames[i+1], dtype=np.float32)
        curr_frame = np.array(base_frames[i], dtype=np.float32)
        for t in range(1, fade_steps+1):
            alpha = t / (fade_steps + 1)
            interp_frame = (1-alpha)*curr_frame + alpha*next_frame
            interp_img = Image.fromarray(np.uint8(interp_frame))
            output_frames.append(interp_img)

# Save GIF
imageio.mimsave(gif_path, output_frames, duration=frame_duration, loop=0)

print(f"Smooth GIF saved as {gif_path}")


Smooth GIF saved as urmia_embeddings_cluster_smooth_joint.gif
